In [1]:
import sqlite3
import pandas as pd

db_path = "data/data.db"

conn = sqlite3.connect(db_path)

df = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)

print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

df.head()

Number of rows: 397711
Number of columns: 8


,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


In [2]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate ticker-date rows:")
print(df.duplicated(subset=["ticker", "ts"]).sum())

print("\nNumber of unique tickers:")
print(df["ticker"].nunique())

Missing values:
ts               0
open             0
high             0
low              0
close            0
adj_close    19855
volume           0
ticker           0
dtype: int64

Duplicate ticker-date rows:
29460

Number of unique tickers:
150


In [3]:
#records that share the same ticker and date
duplicates = df[df.duplicated(subset=["ticker", "ts"], keep=False)].sort_values(["ticker", "ts"])

print("Total rows involved in duplicate groups:", len(duplicates))
duplicates.head(10)

Total rows involved in duplicate groups: 58920


,ts,open,high,low,close,adj_close,volume,ticker
86681,1983-11-15,0.08817,0.088728,0.084821,0.08817,1.483332,118630516,"<a target=""_blank""><span style=""color: blue"">A..."
135214,1983-11-15,0.08817,0.088728,0.084821,0.08817,1.483332,118630516,"<a target=""_blank""><span style=""color: blue"">A..."
195675,1988-06-13,0.39732,0.404020,0.395090,0.40179,0.312510,149609748,"<a target=""_blank""><span style=""color: blue"">A..."
322192,1988-06-13,0.39732,0.404020,0.395090,0.40179,0.312510,149609748,"<a target=""_blank""><span style=""color: blue"">A..."
356732,1990-03-29,0.36830,0.370540,0.363840,0.36719,0.290640,96891296,"<a target=""_blank""><span style=""color: blue"">A..."
383239,1990-03-29,0.36830,0.370540,0.363840,0.36719,0.290640,96891296,"<a target=""_blank""><span style=""color: blue"">A..."
137663,1991-08-07,0.44196,0.455360,0.440850,0.44978,0.360960,211613008,"<a target=""_blank""><span style=""color: blue"">A..."
150940,1991-08-07,0.44196,0.455360,0.440850,0.44978,0.360960,211613008,"<a target=""_blank""><span style=""color: blue"">A..."
312013,1991-10-10,0.42857,0.437500,0.417410,0.42634,0.342920,157214556,"<a target=""_blank""><span style=""color: blue"">A..."
356191,1991-10-10,0.42857,0.437500,0.417410,0.42634,0.342920,157214556,"<a target=""_blank""><span style=""color: blue"">A..."


In [4]:
print("Completely identical rows:", df.duplicated().sum())

print("Duplicate ticker-date rows:", df.duplicated(subset=["ticker", "ts"]).sum())

print("\nSample ticker values:")
for ticker in df["ticker"].drop_duplicates().head(10):
    print(repr(ticker))

Completely identical rows: 29460
Duplicate ticker-date rows: 29460

Sample ticker values:
'CAT'
'CVX'
'HD'
'AAPL'
'TRV'
'JNJ'
'DIS'
'<a target="_blank"><span style="color: blue">CSCO</span></a>'
'SHW'
'KO'


In [5]:
import re

#copy of original data
cleaned_df = df.copy()

cleaned_df = cleaned_df.drop_duplicates()
# Remove HTML tags 
cleaned_df["ticker"] = cleaned_df["ticker"].apply(lambda x: re.sub(r"<[^>]+>", "", x).strip())
cleaned_df = cleaned_df.dropna(subset=["adj_close"])

#dates to datetime
cleaned_df["ts"] = pd.to_datetime(cleaned_df["ts"])

cleaned_df = cleaned_df.sort_values(["ticker", "ts"]).reset_index(drop=True)

print("Original rows:", len(df))
print("Cleaned rows:", len(cleaned_df))
print("Remaining missing adjusted prices:", cleaned_df["adj_close"].isna().sum())
print("Remaining duplicate ticker-date rows:", cleaned_df.duplicated(subset=["ticker", "ts"]).sum())

cleaned_df.head()

Original rows: 397711
Cleaned rows: 349877
Remaining missing adjusted prices: 0
Remaining duplicate ticker-date rows: 0


,ts,open,high,low,close,adj_close,volume,ticker
0,1980-12-12,0.12835,0.12835,0.12835,0.12835,0.098941,469034068,AAPL
1,1980-12-15,0.12165,0.12165,0.12165,0.12165,0.093779,175884971,AAPL
2,1980-12-16,0.11272,0.11272,0.11272,0.11272,0.086896,105728103,AAPL
3,1980-12-17,0.11551,0.11551,0.11551,0.11551,0.089046,86441684,AAPL
4,1980-12-18,0.11886,0.11886,0.11886,0.11886,0.091627,73449671,AAPL


In [6]:
# Inspect extreme adjusted-close values without removing records
anomalies = cleaned_df[
    cleaned_df["adj_close"] > cleaned_df["close"] * 5
].copy()

print("Number of flagged records:", len(anomalies))
print("Number of affected tickers:", anomalies["ticker"].nunique())

Number of flagged records: 7250
Number of affected tickers: 30


In [7]:
# Examining records flagged by the five-times rule
print("Number of affected tickers:", anomalies["ticker"].nunique())

print("\nTickers with the most flagged records:")
print(anomalies["ticker"].value_counts().head(15))

print("\nSample flagged records:")
display(anomalies[["ts", "ticker", "close", "adj_close"]].head(10))

Number of affected tickers: 30

Tickers with the most flagged records:
ticker
DIS     344
BA      334
MCD     329
WMT     326
AXP     310
AAPL    302
JNJ     300
CAT     298
SHW     296
IBM     267
AMGN    260
HD      258
UNH     253
HON     250
MSFT    250
Name: count, dtype: int64

Sample flagged records:


,ts,ticker,close,adj_close
33,1981-02-03,AAPL,0.123330,1.264587
89,1981-04-28,AAPL,0.126120,2.036653
97,1981-05-08,AAPL,0.125000,2.129290
155,1981-08-06,AAPL,0.112720,2.009828
159,1981-08-13,AAPL,0.103800,1.122224
160,1981-08-14,AAPL,0.102120,1.171347
197,1981-10-08,AAPL,0.082589,1.245215
250,1981-12-31,AAPL,0.098772,1.730921
259,1982-01-15,AAPL,0.089286,1.577125
280,1982-02-16,AAPL,0.082031,1.482140


In [2]:
# # Saving stock data to a new table
# cleaned_df.to_sql("ohlc_hw_cleaned",conn, if_exists="replace",index=False)
# #verifying
# result = pd.read_sql_query("SELECT COUNT(*) AS row_count FROM ohlc_hw_cleaned", conn)

# print("Rows saved to ohlc_hw_cleaned:", result["row_count"].iloc[0])

NameError: name 'cleaned_df' is not defined

In [8]:
# Inspect the three suspicious AAPL records
dates_to_check = pd.to_datetime([
    "2024-06-27",
    "2024-08-22",
    "2024-09-17"
])

cleaned_df.loc[
    (cleaned_df["ticker"] == "AAPL") &
    (cleaned_df["ts"].isin(dates_to_check)),
    ["ts", "ticker", "close", "adj_close"]
]

,ts,ticker,close,adj_close
10399,2024-06-27,AAPL,214.10,5109.148400
10435,2024-08-22,AAPL,224.53,1455.309958
10451,2024-09-17,AAPL,216.79,1301.927946


In [9]:
# Remove the three confirmed suspicious AAPL records
bad_dates = pd.to_datetime([
    "2024-06-27",
    "2024-08-22",
    "2024-09-17"
])

bad_rows = (
    (cleaned_df["ticker"] == "AAPL") &
    (cleaned_df["ts"].isin(bad_dates))
)

print("Records to remove:", bad_rows.sum())

cleaned_df = cleaned_df.loc[~bad_rows].copy()

print("Cleaned rows remaining:", len(cleaned_df))

Records to remove: 3
Cleaned rows remaining: 349874


In [11]:
# Saving stock data to a new table
cleaned_df.to_sql("ohlc_hw_cleaned",conn, if_exists="replace",index=False)
#verifying
result = pd.read_sql_query("SELECT COUNT(*) AS row_count FROM ohlc_hw_cleaned", conn)

print("Rows saved to ohlc_hw_cleaned:", result["row_count"].iloc[0])

Rows saved to ohlc_hw_cleaned: 349874
